# Interactive Dashboard for Segmenting "Credit-Fearful" Households Using Dash, K-Means, and PCA (2019 SCF Data)

## Latihan konsep Dash dan data dummy

1. instalasi dash (tidak dieksekusi)

In [1]:
# %pip install dash

2. Membuat aplikasi Dash minimal (Dash(__name__)) dengan layout html.Div berisi satu H1. Aplikasi belum dijalankan.

In [2]:
from dash import Dash, html

# Create a minimal app (we won't run it yet)
demo_app = Dash(__name__)
demo_app.layout = html.Div([
    html.H1("Hello, Dash!")
])
# To run: demo_app.run(host="0.0.0.0", jupyter_mode="inline")
print("Dash app created successfully. Layout contains:", type(demo_app.layout))

Dash app created successfully. Layout contains: <class 'dash.html.Div.Div'>


3. Layout bersarang (H1, H2, P, dan Div berisi H3 dan P). Root berupa Div dengan 4 anak.

In [3]:
from dash import html

# Build a nested layout structure
sample_layout = html.Div([
    html.H1("Main Title"),
    html.H2("Section 1"),
    html.P("This is a paragraph inside section 1."),
    html.Div([
        html.H3("Subsection 1.1"),
        html.P("Nested content goes here.")
    ], id="subsection-container")
])

# Inspect the structure
print("Root element:", type(sample_layout).__name__)
print("Number of children:", len(sample_layout.children))
print("First child type:", type(sample_layout.children[0]).__name__)

Root element: Div
Number of children: 4
First child type: H1


4. Komponen interaktif: dcc.RadioItems (3 opsi, default “a”) dan dcc.Slider (rentang 2–10, default 5).

In [4]:
from dash import dcc

# RadioItems example
radio = dcc.RadioItems(
    options=[
        {"label": "Option A", "value": "a"},
        {"label": "Option B", "value": "b"},
        {"label": "Option C", "value": "c"}
    ],
    value="a",  # Default selection
    id="demo-radio"
)
print("RadioItems default value:", radio.value)
print("Number of options:", len(radio.options))

# Slider example
slider = dcc.Slider(
    min=2,
    max=10,
    step=1,
    value=5,  # Default position
    id="demo-slider"
)
print("Slider range:", slider.min, "to", slider.max)
print("Slider default value:", slider.value)

RadioItems default value: a
Number of options: 3
Slider range: 2 to 10
Slider default value: 5


5. Membuat bar chart Plotly lalu membungkusnya dengan dcc.Graph (id sample-chart).

In [5]:
import plotly.express as px
from dash import dcc

# Create a simple Plotly bar chart
sample_data = {"Feature": ["A", "B", "C"], "Value": [10, 25, 15]}
fig = px.bar(x=sample_data["Value"], y=sample_data["Feature"], orientation="h")
fig.update_layout(xaxis_title="Value", yaxis_title="Feature", height=200)

# Wrap it in a dcc.Graph component
graph_component = dcc.Graph(id="sample-chart", figure=fig)
print("Graph component ID:", graph_component.id)
print("Figure has", len(fig.data), "trace(s)")

# Display the figure directly
fig.show()

Graph component ID: sample-chart
Figure has 1 trace(s)


6. Demo callback: slider (Input) memperbarui teks (Output). Fungsi update_text(7) diuji langsung.

In [6]:
from dash import Dash, Input, Output, dcc, html

# Create a demo app with a slider and text output
callback_demo = Dash(__name__)
callback_demo.layout = html.Div([
    html.H3("Callback Demo"),
    dcc.Slider(min=1, max=10, step=1, value=5, id="input-slider"),
    html.Div(id="output-text")
])

# Define a callback: when slider changes, update the text
@callback_demo.callback(
    Output("output-text", "children"),
    Input("input-slider", "value")
)
def update_text(slider_value):
    return f"The slider value is: {slider_value}"

# Test the callback function directly (without running the server)
result = update_text(7)
print("Callback output for slider_value=7:", result)

Callback output for slider_value=7: The slider value is: 7


7. Contoh callback dengan dua input (radio dan slider), diuji dengan nilai contoh.

In [7]:
from dash import Input, Output

# Example with multiple inputs (conceptual - not connected to an app)
def demo_multi_input(radio_value, slider_value):
    """Callback with two inputs."""
    return f"Radio: {radio_value}, Slider: {slider_value}"

# Test with sample values
print(demo_multi_input("trimmed", 5))
print(demo_multi_input("not_trimmed", 8))

Radio: trimmed, Slider: 5
Radio: not_trimmed, Slider: 8


8. Pipeline StandardScaler + KMeans(k=3) pada 8 titik mainan. Inertia 0,32 dan silhouette 0,864.

In [8]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Create toy data for demonstration
toy_data = pd.DataFrame({
    "feature_1": [1, 2, 3, 10, 11, 12, 5, 6],
    "feature_2": [2, 3, 2, 11, 12, 11, 20, 21]
})

# Build a pipeline: scale first, then cluster
pipeline = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=3, random_state=42, n_init=10)
)
pipeline.fit(toy_data)

# Access the fitted KMeans model
kmeans = pipeline.named_steps["kmeans"]
print("Cluster labels:", kmeans.labels_)
print("Inertia:", round(kmeans.inertia_, 2))

# Compute silhouette score
ss = silhouette_score(toy_data, kmeans.labels_)
print("Silhouette score:", round(ss, 3))

Cluster labels: [0 0 0 2 2 2 1 1]
Inertia: 0.32
Silhouette score: 0.864


9. PCA 2 komponen pada data mainan, hasilnya digabung dengan label klaster.

In [9]:
from sklearn.decomposition import PCA

# Apply PCA to reduce to 2 dimensions
pca = PCA(n_components=2)
toy_pca = pca.fit_transform(toy_data)

# Create a DataFrame with PCA coordinates and labels
pca_df = pd.DataFrame(toy_pca, columns=["PC1", "PC2"])
pca_df["label"] = kmeans.labels_.astype(str)
print(pca_df)

         PC1       PC2 label
0  -9.475654 -2.415986     0
1  -8.213710 -1.777630     0
2  -8.852066 -0.515687     0
3   1.881840  3.329216     2
4   3.143783  3.967572     2
5   2.505428  5.229516     2
6   8.874218 -4.227678     1
7  10.136161 -3.589322     1


10. Perbandingan variance biasa (1303,47) dan trimmed variance (1086,22) pada data dengan outlier.

In [10]:
import pandas as pd
from scipy.stats.mstats import trimmed_var

# Create toy data with an outlier
toy_series = pd.Series([10, 12, 11, 13, 12, 100])  # 100 is an outlier

# Regular variance (affected by outlier)
regular_var = toy_series.var()
print(f"Regular variance: {regular_var:.2f}")

# Trimmed variance (10% from each tail removed)
trimmed = trimmed_var(toy_series, limits=(0.1, 0.1))
print(f"Trimmed variance (10% limits): {trimmed:.2f}")

# Apply to a DataFrame: compute trimmed variance for each column
toy_df = pd.DataFrame({
    "col_a": [10, 12, 11, 13, 12, 100],
    "col_b": [5, 6, 5, 7, 6, 8]
})
variances = toy_df.apply(trimmed_var, limits=(0.1, 0.1))
print("\nTrimmed variances by column:")
print(variances.sort_values(ascending=False))

Regular variance: 1303.47
Trimmed variance (10% limits): 1086.22

Trimmed variances by column:
col_a    1086.222222
col_b       1.138889
dtype: float64


11. Mengecek bahwa semua ID di callback (input dan output) ada di layout.

In [11]:
# Example: verifying callback IDs match layout IDs
layout_ids = {"bar-chart", "trim-button", "k-slider", "metrics", "pca-scatter"}
callback_outputs = {"bar-chart", "metrics", "pca-scatter"}
callback_inputs = {"trim-button", "k-slider"}

# Check all callback references exist in layout
missing = (callback_outputs | callback_inputs) - layout_ids
if missing:
    print(f"Warning: These IDs are missing from layout: {missing}")
else:
    print("All callback IDs are present in layout.")

All callback IDs are present in layout.


## Proyek utama dengan data SCF 2019

1. Mengimpor pandas, plotly, Dash, trimmed_var, KMeans, PCA, silhouette_score, make_pipeline, dan StandardScaler.

In [12]:
import pandas as pd
import plotly.express as px
from dash import Dash, Input, Output, dcc, html
from scipy.stats.mstats import trimmed_var
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


2. Fungsi wrangle() membaca CSV lalu memfilter TURNFEAR == 1 dan NETWORTH < 2 juta.

In [13]:
def wrangle(filepath):
    """Read SCF data file into DataFrame.

    Returns only credit-fearful households whose net worth is less than
    $2 million.

    Parameters
    ----------
    filepath : str
        Location of CSV file.

    Returns
    -------
    pd.DataFrame
    """
    df = pd.read_csv(filepath)
    mask = (df["TURNFEAR"] == 1) & (df["NETWORTH"] < 2e6)
    df = df[mask]
    return df

3. Memuat data. Hasilnya 4.418 baris × 351 kolom.

In [14]:
df = wrangle("data/SCFP2019.csv.gz")
print(df.shape)
df.head()

(4418, 351)


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,...,NWCAT,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT
5,2,21,3790.476607,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
6,2,22,3798.868505,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,3,2,2
7,2,23,3799.468393,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
8,2,24,3788.076005,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2
9,2,25,3793.066589,1,50,3,8,2,1,3,...,1,2,1,2,1,1,4,4,2,2


4. assert validasi data (tidak kosong, filter benar).

In [15]:
assert df.shape[0] > 0, "DataFrame is empty after filtering."
assert "TURNFEAR" in df.columns, "Expected column 'TURNFEAR' not found."
assert (df["TURNFEAR"] == 1).all(), (
    "Not all rows have TURNFEAR == 1. "
    f"Found {(df['TURNFEAR'] != 1).sum()} rows with other values."
)
assert (df["NETWORTH"] < 2e6).all(), (
    "Not all rows have NETWORTH < 2,000,000. "
    f"Max NETWORTH found: {df['NETWORTH'].max():,.0f}"
)
print(f"Data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns.")

Data loaded successfully: 4418 rows, 351 columns.


5. Membuat objek app = Dash(__name__).

In [16]:
# Instantiate the Dash application
app = Dash(__name__)

6. Mendefinisikan layout: judul, bar chart (bar-chart), radio trimmed/not trimmed (trim-button), slider k 2–12 (k-slider), area metrik (metrics), dan scatter PCA (pca-scatter).

In [17]:
# Define the application layout
app.layout = html.Div(
    [
        # Headers
        html.H1("Survey of Consumer Finances"),
        html.H2("High Variance Features"),
        # Bar chart
        dcc.Graph(id="bar-chart"),
        # Radio buttons for trimmed variance
        dcc.RadioItems(
            options=[
                {"label": "trimmed", "value": True},
                {"label": "not trimmed", "value": False},
            ],
            value=True,
            id="trim-button",
        ),
        # K-means section headers
        html.H2("K-means Clustering"),
        html.H3("Number of Clusters (k)"),
        # Slider for k
        dcc.Slider(min=2, max=12, step=1, value=2, id="k-slider"),
        # Metrics display
        html.Div(id="metrics"),
        # PCA scatter plot
        dcc.Graph(id="pca-scatter"),
    ]
)

7. assert memeriksa layout memiliki minimal 9 komponen.

In [18]:
# Verify layout structure
assert app.layout is not None, "app.layout is not defined."
layout_children = app.layout.children
assert len(layout_children) >= 9, (
    f"Expected at least 9 children in layout, got {len(layout_children)}."
)
print("Layout defined successfully with required components.")

Layout defined successfully with required components.


8. get_high_var_features() mengembalikan 5 fitur dengan variance terbesar, baik trimmed maupun biasa, sebagai Series atau daftar nama.

In [19]:
def get_high_var_features(trimmed=True, return_feat_names=False):
    """Return the five highest-variance features of df.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, calculates trimmed variance, removing bottom and top 10%
        of observations.
    return_feat_names : bool, default=False
        If True, returns feature names as a list. If False, returns a Series
        where index is feature names and values are variances.

    Returns
    -------
    pd.Series or list
    """
    if trimmed:
        top_five = (
            df.apply(trimmed_var, limits=(0.1, 0.1)).sort_values().tail(5)
        )
    else:
        top_five = df.var().sort_values().tail(5)

    if return_feat_names:
        top_five = top_five.index.tolist()

    return top_five

9. Callback serve_bar_chart: radio memperbarui bar chart horizontal variance.

In [20]:
@app.callback(Output("bar-chart", "figure"), Input("trim-button", "value"))
def serve_bar_chart(trimmed=True):
    """Return a horizontal bar chart of the five highest-variance features.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, calculates trimmed variance.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    top_five = get_high_var_features(trimmed, return_feat_names=False)
    fig = px.bar(x=top_five, y=top_five.index, orientation="h")
    fig.update_layout(xaxis_title="Variance", yaxis_title="Feature")
    return fig

10. Callback serve_bar_chart: radio memperbarui bar chart horizontal variance.

In [21]:
# Test get_high_var_features
test_features = get_high_var_features(trimmed=True, return_feat_names=True)
assert isinstance(test_features, list), (
    f"Expected list, got {type(test_features).__name__}."
)
assert len(test_features) == 5, (
    f"Expected 5 features, got {len(test_features)}."
)

# Test serve_bar_chart returns a figure
test_fig = serve_bar_chart(trimmed=True)
assert hasattr(test_fig, "data"), "serve_bar_chart did not return a valid figure."
print(f"High variance features (trimmed): {test_features}")
print("Bar chart function working correctly.")

High variance features (trimmed): ['DEBT', 'NETWORTH', 'HOUSES', 'NFIN', 'ASSET']
Bar chart function working correctly.


11. get_model_metrics() membangun pipeline scaler+KMeans dan mengembalikan model, atau inertia dan silhouette.

In [22]:
def get_model_metrics(trimmed=True, k=2, return_metrics=False):
    """Build a KMeans model based on the five highest-variance features.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, uses trimmed variance for feature selection.
    k : int, default=2
        Number of clusters.
    return_metrics : bool, default=False
        If False, returns the fitted KMeans pipeline. If True, returns a dict
        with 'inertia' and 'silhouette' keys.

    Returns
    -------
    sklearn.pipeline.Pipeline or dict
    """
    features = get_high_var_features(trimmed=trimmed, return_feat_names=True)
    X = df[features]
    model = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42)
    )
    model.fit(X)

    if return_metrics:
        inertia = model.named_steps["kmeans"].inertia_
        ss = silhouette_score(X, model.named_steps["kmeans"].labels_)
        return {"inertia": round(inertia), "silhouette": round(ss, 3)}

    return model

12. Callback serve_metrics: menampilkan inertia dan silhouette sebagai elemen H3.

In [23]:
@app.callback(
    Output("metrics", "children"),
    Input("trim-button", "value"),
    Input("k-slider", "value"),
)
def serve_metrics(trimmed=True, k=2):
    """Return H3 elements displaying inertia and silhouette score.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, uses trimmed variance for feature selection.
    k : int, default=2
        Number of clusters.

    Returns
    -------
    list of html.H3
    """
    metrics = get_model_metrics(trimmed, k, return_metrics=True)
    text = [
        html.H3(f"Inertia: {metrics['inertia']}"),
        html.H3(f"Silhouette Score: {metrics['silhouette']}"),
    ]
    return text

13. Pengujian: k = 3 memberi inertia 7.191 dan silhouette 0,704.

In [24]:
# Test get_model_metrics
test_model = get_model_metrics(trimmed=True, k=3, return_metrics=False)
assert hasattr(test_model, "named_steps"), (
    "get_model_metrics should return a Pipeline."
)
assert "kmeans" in test_model.named_steps, (
    "Pipeline should contain a 'kmeans' step."
)

test_metrics = get_model_metrics(trimmed=True, k=3, return_metrics=True)
assert "inertia" in test_metrics, "Metrics dict missing 'inertia' key."
assert "silhouette" in test_metrics, "Metrics dict missing 'silhouette' key."
assert -1 <= test_metrics["silhouette"] <= 1, (
    f"Silhouette score should be in [-1, 1], got {test_metrics['silhouette']}."
)
print(f"Model metrics (k=3): {test_metrics}")
print("Metrics functions working correctly.")

Model metrics (k=3): {'inertia': 7191, 'silhouette': 0.704}
Metrics functions working correctly.


14. get_pca_labels() menjalankan PCA 2 komponen lalu menambahkan label klaster.

In [25]:
def get_pca_labels(trimmed=True, k=2):
    """Return a DataFrame with PCA coordinates and KMeans cluster labels.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, uses trimmed variance for feature selection.
    k : int, default=2
        Number of clusters.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns 'PC1', 'PC2', and 'labels'.
    """
    # Create feature matrix from top 5 features
    features = get_high_var_features(trimmed=trimmed, return_feat_names=True)
    X = df[features]

    # Apply PCA to reduce to 2 dimensions
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    X_pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])

    # Get cluster labels from fitted model
    model = get_model_metrics(trimmed, k, return_metrics=False)
    X_pca_df["labels"] = model.named_steps["kmeans"].labels_.astype(object)
    X_pca_df.sort_values("labels", inplace=True)

    return X_pca_df


15. Callback serve_scatter_plot: scatter plot PCA berwarna per klaster, bereaksi terhadap radio dan slider.

In [26]:
@app.callback(
    Output("pca-scatter", "figure"),
    Input("trim-button", "value"),
    Input("k-slider", "value"),
)
def serve_scatter_plot(trimmed=True, k=2):
    """Return a 2D scatter plot of PCA-reduced data with cluster colors.

    Parameters
    ----------
    trimmed : bool, default=True
        If True, uses trimmed variance for feature selection.
    k : int, default=2
        Number of clusters.

    Returns
    -------
    plotly.graph_objects.Figure
    """
    fig = px.scatter(
        data_frame=get_pca_labels(trimmed, k),
        x="PC1",
        y="PC2",
        color="labels",
        title="PCA Representation of Clusters",
    )
    fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
    return fig

16. Pengujian PCA (bentuk 4.418 × 3, jumlah label sesuai k).

In [27]:
# Test get_pca_labels
test_pca_df = get_pca_labels(trimmed=True, k=3)
assert isinstance(test_pca_df, pd.DataFrame), (
    f"Expected DataFrame, got {type(test_pca_df).__name__}."
)
expected_cols = {"PC1", "PC2", "labels"}
assert expected_cols.issubset(test_pca_df.columns), (
    f"Missing columns. Expected {expected_cols}, got {set(test_pca_df.columns)}."
)
n_unique_labels = test_pca_df["labels"].nunique()
assert n_unique_labels == 3, (
    f"Expected 3 unique labels for k=3, got {n_unique_labels}."
)

# Test serve_scatter_plot
test_scatter = serve_scatter_plot(trimmed=True, k=3)
assert hasattr(test_scatter, "data"), (
    "serve_scatter_plot did not return a valid figure."
)
print(f"PCA DataFrame shape: {test_pca_df.shape}")
print("PCA scatter plot function working correctly.")

PCA DataFrame shape: (4418, 3)
PCA scatter plot function working correctly.


17. app.run(...) masih dikomentari, jadi server belum dijalankan dari sel ini.

In [31]:
app.run(host="127.0.0.1", port=9000, debug=False)

18. webbrowser.open membuka http://127.0.0.1:9000.

In [32]:
import webbrowser

webbrowser.open("http://127.0.0.1:9000")

True

[2026-10-31 22:19:36,459] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "c:\Users\jason\.conda\envs\ds-project\Lib\site-packages\flask\app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jason\.conda\envs\ds-project\Lib\site-packages\flask\app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jason\.conda\envs\ds-project\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jason\.conda\envs\ds-project\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\jason\.c

19. Pesan pengingat untuk menguji dasbor secara manual (radio dan slider).

In [30]:
# No programmatic checkpoint for app deployment.
# Verify the app is running by interacting with the dashboard:
# 1. Toggle the radio button and confirm the bar chart updates.
# 2. Move the slider and confirm the metrics and scatter plot update.
print("Application deployment complete. Interact with the dashboard to verify.")

Application deployment complete. Interact with the dashboard to verify.


Kesimpulan Singkat

- Tujuan: 
    - Mengubah analisis klastering sebelumnya menjadi dasbor web interaktif yang menampilkan fitur berkeragaman tinggi, metrik model, dan visualisasi PCA untuk rumah tangga credit fearful dengan net worth < $2 juta (4.418 rumah tangga).
- Arsitektur: 
    - Satu layout dengan 3 output (bar-chart, metrics, pca-scatter) dan 2 input (trim-button, k-slider). 
    - Logika dipisah menjadi fungsi kecil yang bisa diuji, sehingga lebih rapi dan mudah dipelihara.
- Interaktivitas: 
    - Pengguna dapat memilih variance trimmed atau biasa dan mengubah jumlah klaster (2–12). 
    - Setelah itu bar chart, inertia, silhouette, dan scatter PCA langsung diperbarui.
- Hasil: 
    - Dengan trimmed variance, lima fitur terpilih adalah DEBT, NETWORTH, HOUSES, NFIN, ASSET. 
    - Untuk k = 3, inertia 7.191 dan silhouette 0,704.

Catatan kritis:
- PCA di get_pca_labels() masih dijalankan pada data belum diskalakan, padahal KMeans memakai data yang diskalakan. Sebaiknya PCA dijalankan pada data yang sudah diskalakan.
silhouette_score juga dihitung pada X asli, bukan data hasil scaling.
- get_high_var_features() memakai df global, dan get_model_metrics() dipanggil ulang pada setiap interaksi. Hasilnya bisa di-cache supaya dasbor lebih cepat.